# Flood mapping from Sentinel-1, trained and scored on Copernicus EMS

Every flood Copernicus EMS mapped in Italy becomes a label; the nearest open Sentinel-1 pass becomes the image. This notebook:

1. asks the Copernicus Data Space for Sentinel-1 backscatter **on exactly the grid of each EMS label** (`copernicus-ems pairs`), within a budget of processing units;
2. cuts the pairs into a dataset whose train / val / test are split **by area**, so no ground is in two splits (`copernicus-ems dataset`);
3. trains two flood detectors that run on any laptop: a **threshold on VV** (the classic Sentinel-1 baseline) and a **per-pixel classifier** (scikit-learn);
4. scores both on the test areas **only where EMS observed**: a prediction outside every AOI is reported, never counted as a false alarm (`copernicus_ems.ml.evaluate`).

It needs a free [Copernicus Data Space](https://dataspace.copernicus.eu) account for step 1 (`CDSE_CLIENT_ID`, `CDSE_CLIENT_SECRET`). Images are kept in the data directory: a second run spends nothing.

In [ ]:
import json
import os
from pathlib import Path

import numpy as np

# Italian floods mapped by Rapid Mapping since 2023 whose products have a Sentinel-1 pass
# covering 90% of the AOI within 24 h: about 94 PU at 40 m (a general account has 10,000 a month).
ACTIVATIONS = [
    "EMSR659",
    "EMSR664",
    "EMSR705",
    "EMSR762",
    "EMSR795",
    "EMSR800",
    "EMSR843",
    "EMSR855",
]
RESOLUTION_M = 40  # metres per pixel: 20 m costs four times the PU
BUDGET_PU = float(os.environ.get("NOTEBOOK_BUDGET_PU", 100))  # nothing is sent above it
CHIP = 128  # 128 px x 40 m = 5.1 km
SEED = 0

FETCH = os.environ.get("NOTEBOOK_FETCH", "1") == "1"  # 0: use the pairs already in DATA_DIR
DATA_DIR = Path(os.environ.get("COPERNICUS_EMS_DATA_DIR", "~/.cache/copernicus-ems")).expanduser()
DATASET = DATA_DIR / "datasets" / f"italy-floods-s1-{RESOLUTION_M}m"
FIGURE = Path(os.environ.get("NOTEBOOK_FIGURE", "../docs/images/notebook-flood-s1.png"))
print("data:", DATA_DIR, "| dataset:", DATASET)

## 1. Image–label pairs

For each product, `plan_pair` picks the Sentinel-1 pass nearest to the time EMS observed the flood, among those covering 90% of the AOI within ±24 h, and prices it. Nothing is sent before the whole plan fits the budget. Products of one AOI seen by the same pass share one image.

In [ ]:
from copernicus_ems import RapidMapping
from copernicus_ems.sentinelhub import SentinelHub, make_pair, plan_pair

if FETCH:
    plans = []
    async with RapidMapping(directory=DATA_DIR) as ems:
        for code in ACTIVATIONS:
            for event in await ems.observed_events(
                code
            ):  # floods only: the features become the label
                if event.product.observed_time and event.product.aoi_extent:
                    plans.append(await plan_pair(ems, event, resolution_m=RESOLUTION_M))
    chosen = [p for p in plans if p.scene_pass is not None]
    to_fetch = {p.backscatter_path: p for p in chosen if not p.backscatter_path.is_file()}
    cost = sum(p.pu for p in to_fetch.values())
    print(
        f"{len(plans)} products, {len(chosen)} with a pass, {len(to_fetch)} images to fetch, {cost:.1f} PU"
    )
    assert cost <= BUDGET_PU, f"{cost:.1f} PU is over the budget of {BUDGET_PU}"
    async with SentinelHub() as hub:
        for plan in chosen:
            await make_pair(hub, plan)
    print(f"spent about {hub.pu_estimated:.1f} PU")
    for p in plans:
        if p.scene_pass is None:
            print(f"  skipped {p.event.product.activation_code} {p.product}: {p.reason}")

## 2. A dataset split by area

The products of one AOI (first delineation, monitoring updates, grading) overlap in space. Splitting their chips at random would put the same fields in training and in test, and the scores would be too good. `build_dataset` assigns whole areas (activation + AOI) to a split. Where the image has no data the label becomes 255 (ignored).

In [ ]:
import csv

from copernicus_ems.dataset import build_dataset, find_pairs

pairs = find_pairs(DATA_DIR, ACTIVATIONS)
summary = build_dataset(pairs, DATASET, size=CHIP, min_observed=0.5, seed=SEED)
meta = json.loads((DATASET / "dataset.json").read_text())
rows = list(csv.DictReader((DATASET / "index.csv").open()))
# Which satellite each EMS map was drawn on. When it is Sentinel-1, the label comes from the very
# image the detectors see: scoring against it is partly circular, so it is reported apart.
made_from = {
    (p.activation, p.product): sorted({i["sensor"] for i in p.meta.get("made_from", [])})
    for p in pairs
}
for r in rows:
    r["made_from"] = ", ".join(made_from.get((r["activation"], r["product"]), []))
    r["same_image"] = "Sentinel-1" in r["made_from"]
same = sum(1 for p in pairs if "Sentinel-1" in made_from.get((p.activation, p.product), []))
print(
    f"{summary.pairs} pairs -> {summary.chips} chips of {CHIP} px; {same} pairs have labels drawn on Sentinel-1"
)
for split, info in meta["splits"].items():
    print(
        f"  {split:5} {info['chips']:4d} chips  flooded share {info['flooded_share']}  areas: {', '.join(info['groups'])}"
    )

## 3. Features

Backscatter is linear power; floods are dark (smooth water reflects the radar away from the satellite), so the usual features are in dB. The classifier also gets the local mean and spread of VV, which separate open water from dark but rough surfaces.

In [ ]:
import rasterio
from scipy.ndimage import uniform_filter

from copernicus_ems.ml import FLOODED, IGNORE, LabelRaster


def db(x):
    return 10 * np.log10(np.maximum(x, 1e-6))


def features(path):
    with rasterio.open(path) as src:
        vv, vh, _mask = src.read()
    vv_db, vh_db = db(vv), db(vh)
    mean = uniform_filter(vv_db, 5)
    spread = np.sqrt(np.maximum(uniform_filter(vv_db**2, 5) - mean**2, 0))
    return np.stack([vv_db, vh_db, vv_db - vh_db, mean, spread], axis=-1)


def load(split):
    chosen = [r for r in rows if r["split"] == split]
    x = (
        np.stack([features(DATASET / r["image"]) for r in chosen])
        if chosen
        else np.empty((0, CHIP, CHIP, 5))
    )
    y = [LabelRaster.from_geotiff(DATASET / r["label"]) for r in chosen]
    return chosen, x, y


data = {split: load(split) for split in ("train", "val", "test")}
for split, (chosen, _x, y) in data.items():
    observed = sum(int((label.labels != IGNORE).sum()) for label in y)
    wet = sum(int((label.labels == FLOODED).sum()) for label in y)
    print(
        f"{split:5} {len(chosen):4d} chips  {observed:9d} observed px  {wet / max(observed, 1):.1%} flooded"
    )

## 4. Two detectors

* **Threshold**: water where VV is darker than *t* dB, with *t* chosen on the training areas to maximise the critical success index (CSI).
* **Classifier**: gradient-boosted trees on the five features, trained on a balanced sample of observed training pixels; the probability cut is chosen on the validation areas.

Only observed pixels (labels 0 or 1) are used to fit; 255 never is.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

from copernicus_ems.ml import Scores, evaluate


def pixels(split):
    _, x, y = data[split]
    if not len(y):
        return np.empty((0, 5)), np.empty(0)
    labels = np.stack([label.labels for label in y])
    keep = labels != IGNORE
    return x[keep], (labels[keep] == FLOODED).astype(np.uint8)


def score(split, predict, keep=lambda _row: True):
    chosen, x, y = data[split]
    return sum(
        (evaluate(predict(x[i]), y[i]) for i in range(len(y)) if keep(chosen[i])), Scores.zero()
    )


x_train, y_train = pixels("train")
best_t = max(
    np.arange(-30, -8, 0.25), key=lambda t: score("train", lambda f: f[..., 0] < t).csi or 0
)
print(f"threshold: VV < {best_t:.2f} dB")

rng = np.random.default_rng(SEED)
wet, dry = np.flatnonzero(y_train == 1), np.flatnonzero(y_train == 0)
n = min(len(wet), 100_000)
sample = np.concatenate(
    [rng.choice(wet, n, replace=False), rng.choice(dry, min(len(dry), 3 * n), replace=False)]
)
model = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, random_state=SEED)
model.fit(x_train[sample], y_train[sample])


def probability(f):
    return model.predict_proba(f.reshape(-1, f.shape[-1]))[:, 1].reshape(f.shape[:-1])


best_p = max(
    np.arange(0.1, 0.95, 0.05), key=lambda p: score("val", lambda f: probability(f) > p).csi or 0
)
print(f"classifier: trained on {len(sample):,} pixels, cut at p > {best_p:.2f}")

## 5. Scores on the test areas

Contingency counted **inside the observation masks only**. `unobserved km²` is water predicted outside every AOI: reported, never a false alarm, because nobody looked there.

Each detector is scored three times: on all test areas, on the chips whose EMS map was drawn **on another satellite** (an independent reference), and on those drawn **on Sentinel-1 itself** (the same pixels the detector sees, so an optimistic reference).

In [ ]:
detectors = {
    "threshold": lambda f: f[..., 0] < best_t,
    "classifier": lambda f: probability(f) > best_p,
}
subsets = {
    "all": lambda _row: True,
    "independent": lambda row: not row["same_image"],
    "same image": lambda row: row["same_image"],
}
test_rows = data["test"][0]
print("test chips:", {k: sum(1 for r in test_rows if keep(r)) for k, keep in subsets.items()})
print(
    f"{'':12} {'labels':12} {'CSI':>6} {'POD':>6} {'FAR':>6} {'F1':>6} {'bias':>6} {'unobserved km²':>15}"
)
results = {}
for name, predict in detectors.items():
    for subset, keep in subsets.items():
        s = results[name, subset] = score("test", predict, keep)
        if s.hits + s.misses + s.false_alarms == 0:
            print(f"{name:12} {subset:12}   (no chip)")
            continue
        print(
            f"{name:12} {subset:12} {s.csi:6.3f} {s.pod:6.3f} {s.far:6.3f} {s.f1:6.3f} {s.bias:6.2f} {s.unobserved_predicted_km2:15.1f}"
        )

## 6. What the detectors see

The test chips with the most mapped flood: VV backscatter, EMS label (blue flooded, beige dry, grey not observed), and the two predictions.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

chosen, x_test, y_test = data["test"]
order = sorted(range(len(y_test)), key=lambda i: -int((y_test[i].labels == FLOODED).sum()))[:3]
label_colors = ListedColormap(["#e9dfc8", "#2b6cb0", "#bdbdbd"])

fig, axes = plt.subplots(len(order), 4, figsize=(12, 3.1 * len(order)), squeeze=False)
for row, i in enumerate(order):
    shown = np.where(y_test[i].labels == IGNORE, 2, y_test[i].labels)
    panels = [
        (x_test[i][..., 0], {"cmap": "gray", "vmin": -25, "vmax": 0}, "Sentinel-1 VV (dB)"),
        (
            shown,
            {"cmap": label_colors, "vmin": 0, "vmax": 2, "interpolation": "nearest"},
            "EMS label",
        ),
        (
            detectors["threshold"](x_test[i]),
            {"cmap": "Blues", "vmin": 0, "vmax": 1},
            f"threshold VV < {best_t:.1f} dB",
        ),
        (detectors["classifier"](x_test[i]), {"cmap": "Blues", "vmin": 0, "vmax": 1}, "classifier"),
    ]
    for ax, (image, style, title) in zip(axes[row], panels, strict=True):
        ax.imshow(image, **style)
        ax.set_xticks([]), ax.set_yticks([])
        if row == 0:
            ax.set_title(title, fontsize=10)
    axes[row][0].set_ylabel(
        f"{chosen[i]['activation']} {chosen[i]['aoi']}\ngap {float(chosen[i]['gap_hours']):+.0f} h, map on\n{chosen[i]['made_from']}",
        fontsize=9,
    )
fig.text(
    0.5,
    0.005,
    "Contains modified Copernicus Sentinel data and Copernicus EMS information, © European Union",
    ha="center",
    fontsize=8,
)
fig.tight_layout(rect=(0, 0.02, 1, 1))
FIGURE.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURE, dpi=110)
print("saved", FIGURE)

## Reading the scores

* **The labels are a lower bound.** EMS delineates what analysts saw on (often commercial) images; short urban floods and water under trees are missing. Some "false alarms" inside the AOI may be water nobody mapped.
* **Sometimes the image *is* the map's image.** About a third of the pairs have EMS maps drawn on the same Sentinel-1 acquisition (gap ≈ 0 h): there the label and the input share their pixels, and a VV threshold partly reproduces the analysts' semi-automatic extraction. Those chips are good *training* data, but only the *independent* scores say how a detector would do on a flood nobody has mapped yet.
* **Otherwise the image is not the map's image.** Each pair keeps `gap_hours` between the Sentinel-1 pass and the EMS observation (in `index.csv`): water moves in a day. Filtering the pairs to a smaller gap trades data for cleaner labels.
* **A small dataset.** A few Italian activations, a few hundred chips: enough to compare detectors honestly, not to claim a general model.

## Next: a foundation model

The dataset is already in the shape segmentation models expect: aligned GeoTIFF chips, labels with `255 = ignore`, splits by area. To fine-tune an Earth-observation foundation model (for example IBM–NASA Prithvi-EO-2.0 or TerraMind) with [TerraTorch](https://github.com/IBM/terratorch), point its generic segmentation datamodule at `images/` and `labels/`, set `ignore_index: 255`, and score its test predictions with `copernicus_ems.ml.evaluate` exactly as above, so the numbers stay comparable. That needs a GPU; the two detectors here are the baseline it has to beat.

*Data: Copernicus EMS Rapid Mapping, © European Union; Copernicus Sentinel-1, processed by Sentinel Hub on the Copernicus Data Space Ecosystem.*